In [8]:
import torch
import torch.nn as nn
# одна из крутейших букв
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [9]:
IMG_SIZE = 64  # размер изображения, на который мы будем приводить все картинки
le = LabelEncoder()
# pred_labels = le.inverse_transform(pred.cpu().numpy())
# to convert back
from torch.utils.data import TensorDataset, DataLoader

def get_chinese_data(batch_size, transform_train):
    torch.manual_seed(0)
    np.random.seed(0)

    transform_test = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),        # критично
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)) # отрезок -1, 1 но на чб
    ])

    # Загружаем данные
    data1 = np.load('data/train-1.npy', allow_pickle=True)
    data2 = np.load('data/train-2.npy', allow_pickle=True)
    data3 = np.load('data/train-3.npy', allow_pickle=True)
    data4 = np.load('data/train-4.npy', allow_pickle=True)
    data = np.concatenate([data1, data2, data3, data4])
    images = [x[0] for x in data]
    labels = [x[1] for x in data]

    # Suppose `labels` is your full label list (strings or ints)
    labels = le.fit_transform(labels)
    images_train, images_TMP, labels_train, labels_TMP = train_test_split(
        images, labels, test_size=0.3, stratify=labels, shuffle=True, random_state=42
    ) # stratify - для равномерного распределения, и указываем по чему распределение

    images_val, images_test, labels_val, labels_test = train_test_split(
        images_TMP, labels_TMP, test_size=0.5, stratify=labels_TMP, shuffle=True, random_state=42
    )
    # итак, есть train, val, test, для каждого images и labels

    def apply_transform(img_list, transform):
        return torch.stack([transform(img.astype(np.uint8)) for img in img_list])
    
    images_train_tensor = apply_transform(images_train, transform_train)
    images_val_tensor   = apply_transform(images_val, transform_test)
    images_test_tensor  = apply_transform(images_test, transform_test)

    labels_train_tensor = torch.tensor(labels_train, dtype=torch.long)
    labels_val_tensor   = torch.tensor(labels_val, dtype=torch.long)
    labels_test_tensor  = torch.tensor(labels_test, dtype=torch.long)


    train_dataset = TensorDataset(images_train_tensor, labels_train_tensor)
    val_dataset   = TensorDataset(images_val_tensor, labels_val_tensor)
    test_dataset  = TensorDataset(images_test_tensor, labels_test_tensor)

    train_loader    = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader      = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader     = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

In [10]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),        # критично
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))            # отрезок -1, 1 но на чб
])


train_loader, val_loader, test_loader = get_chinese_data(batch_size=64, transform_train=transform)

In [11]:
n_classes = 1000
pool_size = 16
C_NUM = 32
k = 5
p = 2
class BasicBlockNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.relu = nn.ReLU()


        self.globalshortcut = nn.Conv2d(in_channels=1, out_channels=C_NUM, kernel_size=1)
        self.shortcut1 = nn.Conv2d(in_channels=1, out_channels=C_NUM, kernel_size=1)

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=C_NUM, kernel_size=k, padding=p)
        self.bn1 = nn.BatchNorm2d(C_NUM)

        self.shortcut2 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=1)

        self.conv2 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=k, padding=p)
        self.bn2 = nn.BatchNorm2d(C_NUM)

        self.shortcut3 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=1)

        self.conv3 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=k, padding=p)
        self.bn3 = nn.BatchNorm2d(C_NUM)

        self.shortcut4 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=1)

        self.conv4 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=k, padding=p)
        self.bn4 = nn.BatchNorm2d(C_NUM)

        #self.shortcut5 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=1)

        self.conv5 = nn.Conv2d(in_channels=C_NUM, out_channels=C_NUM, kernel_size=k, padding=p)
        self.bn5 = nn.BatchNorm2d(C_NUM)

        self.pool = nn.AvgPool2d(kernel_size=pool_size)
        self.fc = nn.Linear(C_NUM * (IMG_SIZE//pool_size) * (IMG_SIZE//pool_size), n_classes)  # flatten before FC

    def forward(self, x):
        global_sc = self.globalshortcut(x)
        sc1 = self.shortcut1(x)
        x = self.conv1(x)
        x = self.bn1(x)
        x += sc1
        x = self.relu(x)

        sc2 = self.shortcut2(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x += sc2
        x = self.relu(x)

        sc3 = self.shortcut3(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x += sc3
        x = self.relu(x)

        sc4 = self.shortcut4(x)
        x = self.conv4(x)
        x = self.bn4(x)
        x += sc4
        x = self.relu(x)

        #sc5 = self.shortcut5(x)
        x = self.conv5(x)
        x = self.bn5(x)
        x += global_sc
        x = self.relu(x)
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [12]:
net = BasicBlockNet()
device = torch.device('mps:0' if torch.mps.is_available() else 'cpu')
print(device)
net = net.to(device)

mps:0


In [13]:
n_classes = 1000

def test(model, loader):
    loss_log = []
    acc_log = []
    model.eval()
    for data, target in loader:
        data = data.to(device)
        target = target.to(device)

        output = model(data)
        loss = F.cross_entropy(output, target)
        loss_log.append(loss.item())

        pred = output.argmax(dim=1)
        acc = (pred == target).float().mean()
        acc_log.append(acc.item())

    return np.mean(loss_log), np.mean(acc_log)

def train_epoch(model, optimizer, train_loader):
    loss_log = []
    acc_log = []
    model.train()
    for data, target in train_loader:
        data = data.to(device)
        target = target.to(device)

        # зануляем градиенты
        optimizer.zero_grad()
        # логиты
        output = model(data)
        # кэ
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        loss_log.append(loss.item())
        
        # и акк
        pred = output.argmax(dim=1)
        acc = (pred == target).float().mean()
        acc_log.append(acc.item())
        
    return loss_log, acc_log

def train(model, optimizer, n_epochs, train_loader, val_loader, scheduler=None):
    train_loss_log, train_acc_log, val_loss_log, val_acc_log = [], [], [], []

    for epoch in range(n_epochs):
        train_loss, train_acc = train_epoch(model, optimizer, train_loader)
        val_loss, val_acc = test(model, val_loader)

        train_loss_log.extend(train_loss)
        train_acc_log.extend(train_acc)

        val_loss_log.append(val_loss)
        val_acc_log.append(val_acc)

        print(f"Epoch {epoch+1}/{n_epochs}")
        print(f" train loss: {np.mean(train_loss)}, train acc: {np.mean(train_acc)}")
        print(f" val loss: {val_loss}, val acc: {val_acc}\n")

        if scheduler is not None:
            scheduler.step()

    return train_loss_log, train_acc_log, val_loss_log, val_acc_log

In [14]:
optimizer = optim.SGD(net.parameters(), lr=0.1, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
train_loss_log, train_acc_log, val_loss_log, val_acc_log = train(net, optimizer, 10, train_loader, val_loader, scheduler)

/var/folders/cy/nzn4qdtd0wzb3j2lkypb4wk80000gn/T/ipykernel_92628/1065223216.py:37: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Scalar.cpp:23.)
  loss_log.append(loss.item())


Epoch 1/10
 train loss: 1.2432219650966558, train acc: 0.7314541586604447
 val loss: 0.35707858093226, val acc: 0.9045323074703485

Epoch 2/10
 train loss: 0.25177044543275834, train acc: 0.930946335437826
 val loss: 0.23950173848436218, val acc: 0.9376943479274208

Epoch 3/10
 train loss: 0.13702455221264, train acc: 0.9617545635465276
 val loss: 0.2164773845129316, val acc: 0.9457769343550776

Epoch 4/10
 train loss: 0.08177919195630802, train acc: 0.9776969530606643
 val loss: 0.17531265755703915, val acc: 0.9580751554456166

Epoch 5/10
 train loss: 0.04157687131412424, train acc: 0.9885568213011254
 val loss: 0.14667806789566967, val acc: 0.9672838165573824

Epoch 6/10
 train loss: 0.018146192239938137, train acc: 0.9955308125171561
 val loss: 0.14361527241662037, val acc: 0.9714908770806658

Epoch 7/10
 train loss: 0.0075310093129318054, train acc: 0.9985460129014548
 val loss: 0.12934654794650383, val acc: 0.9757465246239636

Epoch 8/10
 train loss: 0.0033107222355256527, train a

In [15]:
test_loss, test_acc = test(net, test_loader)
print(f"test loss: {test_loss:.4f}, test accuracy: {test_acc:.4f}")

test loss: 0.1304, test accuracy: 0.9763


In [16]:
true_test = np.load("data/test.npy", allow_pickle=True)

In [17]:
def predict(model, image):
    model.eval()
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),        # критично
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)) # отрезок -1, 1 но на чб
    ])
    image = transform(image)
    image = image.to(device)
    output = model(image.unsqueeze(0))
    pred = output.argmax(dim=1)
    return le.inverse_transform(pred.cpu().numpy())[0]

import csv
# Open the file for writing
with open('data/answers.csv', mode='w', newline='') as f:
    writer = csv.writer(f)

    # Write the header
    writer.writerow(["Id", "Category"])

    # Write predictions
    for count, img in enumerate(true_test):
        pred = predict(net, img)
        writer.writerow([count+1, pred])